# Feature 013 — robust FCR bidding walkthrough

This notebook demonstrates the native single-day, non-intraday robust bidder on one deterministic 24-hour problem. It solves the mixed-binary **restriction** and **relaxation**, audits the extracted decisions, hands only the eligible restriction plan to Feature 012, and inspects the bounded Julia/Gurobi fixture.

The question is deliberately narrow: can Open Energy Lab select a charge-positive hourly energy baseline and six symmetric-FCR blocks before activation is known, while preserving the reference formulation's evidence boundaries?

## Setup, terminology, and evidence limits

Run from the repository root after:

    uv sync --group notebook
    uv run jupyter lab

The source notebook is committed with empty outputs. Default execution is deterministic and offline: HiGHS solves one authored problem, the realized activation used later is project-authored, and the normalized reference fixture is a small committed JSON file.

- **Restriction**: conservative formulation; it may emit a robust-labelled FixedFCRBidPlan only after a successful solve, zero modeled slack within tolerance, and an independent residual audit.
- **Relaxation**: optimistic upper-bound formulation; its decisions are diagnostic and can never be presented as robustly deployable.
- **Exact formulation**: optional Gurobi evidence only.
- **Julia comparison**: bounded by three-decimal source precision and missing source solver metadata. It is not proof of identical unrounded decisions, optimality, or complete physical-trace parity.
- **Activation**: evaluation-only information. Feature 013 never receives the realized activation trace while selecting a bid.

In [ ]:
from pathlib import Path

RUN_NATIVE_REFERENCE_BENCHMARK = False
RUN_OPTIONAL_GUROBI_EXACT = False
RUN_EXTERNAL_ACTIVATION_REPLAY = False
EXPORT_FIGURES = False

REFERENCE_BENCHMARK_TIME_LIMIT_SECONDS = 3600.0
OPTIONAL_GUROBI_TIME_LIMIT_SECONDS = 60.0
REFERENCE_FIXTURE_PATH = Path(
    "tests/fixtures/fcr_bidding/"
    "robust_storage_bidding_2023-06-18_restriction_v1.json"
)
EXTERNAL_ACTIVATION_PATH = Path(
    "artifacts/reference/robust_storage_bidding/regulation_signal.csv"
)
EVALUATION_FIXTURE_PATH = Path(
    "tests/fixtures/fcr_bid_evaluation/"
    "robust_storage_bidding_2023-06-18_aggregate_v1.json"
)
FIGURE_EXPORT_DIR = Path("tmp/robust-fcr-bidding-notebook/figures")

In [ ]:
from dataclasses import replace
from datetime import UTC, datetime
import hashlib
import json

import matplotlib.pyplot as plt
import pandas as pd

from examples.robust_fcr_bidding.run_small_case import build_walkthrough_problem
from open_energy_lab.control import (
    RobustFCRSolvePolicy,
    build_robust_fcr_reference_problem,
    compare_robust_fcr_bounds,
    compare_robust_fcr_reference,
    load_robust_fcr_reference_fixture,
    solve_robust_fcr_bid,
)
from open_energy_lab.control.robust_fcr_reference import (
    build_reference_decision_intervals,
)
from open_energy_lab.market import (
    ActivationImportRequest,
    CanonicalInterval,
    import_regulation_activation_trace,
)
from open_energy_lab.simulator import (
    BatteryComponent,
    EvaluationMarketContext,
    EvaluationPriceObservation,
    FCRBidEvaluationRequest,
    MultiRateTimeContract,
    evaluate_fixed_fcr_bid,
)

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
generated_figures = {}

def stable_fingerprint(value):
    payload = json.dumps(
        value,
        allow_nan=False,
        ensure_ascii=False,
        separators=(",", ":"),
        sort_keys=True,
    ).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

## Ownership, signs, and multi-rate mapping

Feature 013 owns bid selection. Feature 012 alone owns accepted-plan replay and settlement. The public power convention is positive for battery charging, whereas the pinned source variable is positive for discharge:

$$
p_h^{\mathrm{OEL}}=-x_h^{0,\mathrm{source}},
\qquad
r_b^{\mathrm{OEL}}=x_b^{r,\mathrm{source}}\ge 0.
$$

For realized activation $a_k\in[-1,1]$, supplied only after bid selection, Feature 012 requests:

$$
p_k^{\mathrm{requested}}=p_{h(k)}^{\mathrm{OEL}}-a_k r_{b(k)}.
$$

Positive activation therefore requests upward discharge (negative OEL power). Each four quarter-hours maps to one hourly energy decision, and each sixteen quarter-hours maps to one four-hour FCR block.

In [ ]:
walkthrough_problem = build_walkthrough_problem()
battery = walkthrough_problem.battery
time_contract = walkthrough_problem.time_contract
recovery_budget = walkthrough_problem.uncertainty.recovery_window
horizon_budget = walkthrough_problem.uncertainty.horizon_wide

problem_summary_table = pd.DataFrame(
    [
        ("planning intervals", len(time_contract.planning_intervals), "count"),
        ("energy decisions", len(time_contract.energy_intervals), "count"),
        ("FCR decisions", len(time_contract.fcr_intervals), "count"),
        ("maximum charge power", battery.maximum_charge_power_kw, "kW"),
        ("maximum discharge power", battery.maximum_discharge_power_kw, "kW"),
        ("initial stored energy", battery.initial_stored_energy_kwh, "kWh"),
        ("stored-energy minimum", battery.minimum_stored_energy_kwh, "kWh"),
        ("stored-energy maximum", battery.maximum_stored_energy_kwh, "kWh"),
        ("charge efficiency", battery.charge_efficiency, "fraction"),
        ("discharge efficiency", battery.discharge_efficiency, "fraction"),
    ],
    columns=["quantity", "value", "unit"],
)
problem_summary_table

In [ ]:
mapping_table = pd.DataFrame(
    [
        {
            "planning interval": index,
            "hourly energy index": time_contract.planning_to_energy[index],
            "four-hour FCR index": time_contract.planning_to_fcr[index],
        }
        for index in range(96)
    ]
)
derivation = horizon_budget.derivation_inputs
uncertainty_table = pd.DataFrame(
    [
        ("recovery activation budget gamma-prime", recovery_budget.recovery_activation_budget_hours, "h"),
        ("recovery window Gamma-prime", recovery_budget.recovery_window_hours, "h"),
        ("horizon T", horizon_budget.horizon_hours, "h"),
        ("full recovery windows", derivation["full_window_count"], "count"),
        ("horizon remainder", derivation["remainder_hours"], "h"),
        ("derived horizon budget gamma", horizon_budget.horizon_activation_budget_hours, "h"),
    ],
    columns=["quantity", "value", "unit"],
)
display(mapping_table.head(20), uncertainty_table)

### Two distinct uncertainty sets

The recovery-window set $\Xi'_{\gamma',\Gamma'}$ and the horizon-wide set $\Xi_\gamma$ are separate immutable contracts. For horizon $T$,

$$
n=\left\lfloor \frac{T}{\Gamma'}\right\rfloor,
\qquad
\gamma=n\gamma' + \min(T-n\Gamma',\gamma').
$$

With $T=24$ h, $\gamma'=0.25$ h, and $\Gamma'=2.25$ h, $n=10$, the remainder is $1.5$ h, and the derived non-intraday horizon budget is $2.75$ h. The two sets are not interchangeable tuning parameters.

In [ ]:
hour_indices = list(range(24))
fcr_block_indices = list(range(6))
price_figure, price_axes = plt.subplots(2, 1, figsize=(10, 6), constrained_layout=True)
price_axes[0].step(
    hour_indices,
    walkthrough_problem.prices.day_ahead_energy_prices_eur_per_kwh,
    where="post",
    label="day-ahead energy price",
)
price_axes[0].set_ylabel("Price (EUR/kWh)")
price_axes[0].set_xlabel("Delivery hour")
price_axes[0].set_title("Authored hourly energy-price input")
price_axes[0].legend()
price_axes[1].bar(
    fcr_block_indices,
    walkthrough_problem.prices.fcr_capacity_prices_eur_per_kw_block,
    width=0.8,
    label="FCR capacity price",
)
price_axes[1].set_ylabel("Price (EUR/kW/block)")
price_axes[1].set_xlabel("Four-hour FCR block")
price_axes[1].set_title("Authored FCR-capacity-price input")
price_axes[1].legend()
generated_figures["input_prices"] = price_figure
price_figure

## Restriction and relaxation on identical inputs

Both formulations receive the same problem fingerprint and the same caller-owned HiGHS policy. For this maximization problem, the conservative restriction must satisfy:

$$
z_{\mathrm{restriction}}\le z_{\mathrm{relaxation}}.
$$

Only the restriction is eligible to emit a plan. Equality here means the authored case happens to close the formulation gap; it does not erase the formulations' different roles.

In [ ]:
restriction_result = solve_robust_fcr_bid(
    walkthrough_problem,
    formulation="restriction",
)
relaxation_result = solve_robust_fcr_bid(
    walkthrough_problem,
    formulation="relaxation",
)
bound_comparison = compare_robust_fcr_bounds(
    restriction_result,
    relaxation_result,
)

assert restriction_result.decisions is not None
assert relaxation_result.decisions is not None
assert restriction_result.deployable_plan is not None

In [ ]:
solver_table = pd.DataFrame(
    [
        {
            "formulation": result.formulation.kind,
            "role": result.formulation.deployability_role,
            "solve status": result.solver.solve_status,
            "termination": result.solver.termination_status,
            "objective (EUR)": result.solver.objective_value_eur,
            "best upper bound (EUR)": result.solver.best_upper_bound_eur,
            "relative gap": result.solver.relative_gap,
            "deployability": result.deployability_status,
            "plan emitted": result.deployable_plan is not None,
            "solver": result.solver.solver_name,
            "solver version": result.solver.solver_version,
        }
        for result in (restriction_result, relaxation_result)
    ]
)
bound_table = pd.DataFrame(
    [bound_comparison.restriction_le_relaxation.to_dict()]
)
display(solver_table, bound_table)

In [ ]:
energy_schedule_table = pd.DataFrame(
    {
        "delivery hour": hour_indices,
        "restriction baseline (kW, charge-positive)": restriction_result.decisions.energy_baseline_charge_positive_kw,
        "relaxation baseline (kW, charge-positive)": relaxation_result.decisions.energy_baseline_charge_positive_kw,
    }
)
fcr_schedule_table = pd.DataFrame(
    {
        "four-hour block": fcr_block_indices,
        "restriction symmetric FCR (kW)": restriction_result.decisions.fcr_capacity_kw,
        "relaxation symmetric FCR (kW)": relaxation_result.decisions.fcr_capacity_kw,
    }
)
audit = restriction_result.feasibility_audit
assert audit is not None
audit_table = pd.DataFrame(
    [
        {
            "equation family": family.family_id,
            "relation": family.relation,
            "unit": family.canonical_unit,
            "checks": family.checked_count,
            "maximum positive violation": family.maximum_positive_violation,
        }
        for family in audit.constraint_families
    ]
)
audit_summary_table = pd.DataFrame(
    [
        ("maximum modeled slack", max(audit.modeled_slacks.values()), "canonical energy units"),
        ("maximum independent residual", audit_table["maximum positive violation"].max(), "family canonical unit"),
        ("deployability threshold", audit.deployability_threshold, "absolute per canonical unit"),
        ("passes deployability audit", audit.passes_deployability_audit, "boolean"),
    ],
    columns=["quantity", "value", "unit"],
)
display(energy_schedule_table, fcr_schedule_table, audit_summary_table)

In [ ]:
schedule_figure, schedule_axes = plt.subplots(2, 1, figsize=(10, 6), constrained_layout=True)
schedule_axes[0].step(
    hour_indices,
    restriction_result.decisions.energy_baseline_charge_positive_kw,
    where="post",
    label="restriction",
)
schedule_axes[0].step(
    hour_indices,
    relaxation_result.decisions.energy_baseline_charge_positive_kw,
    where="post",
    linestyle="--",
    label="relaxation",
)
schedule_axes[0].axhline(0.0, color="black", linewidth=0.8)
schedule_axes[0].set_ylabel("Baseline power (kW, charge +)")
schedule_axes[0].set_xlabel("Delivery hour")
schedule_axes[0].set_title("Hourly energy decisions")
schedule_axes[0].legend()
schedule_axes[1].step(
    fcr_block_indices,
    restriction_result.decisions.fcr_capacity_kw,
    where="post",
    label="restriction",
)
schedule_axes[1].step(
    fcr_block_indices,
    relaxation_result.decisions.fcr_capacity_kw,
    where="post",
    linestyle="--",
    label="relaxation",
)
schedule_axes[1].set_ylabel("Symmetric FCR capacity (kW)")
schedule_axes[1].set_xlabel("Four-hour block")
schedule_axes[1].set_title("Four-hour FCR decisions")
schedule_axes[1].legend()
generated_figures["optimized_schedules"] = schedule_figure
schedule_figure

## Power headroom and stored-energy protection

In the public charge-positive frame, every planning interval must retain both symmetric directions:

$$
-P_{\mathrm{discharge}}\le p_k-r_k,
\qquad
p_k+r_k\le P_{\mathrm{charge}}.
$$

For this deliberately hand-calculable flat-reserve case, efficiencies are one and the baseline is zero. Under only the horizon-wide activation budget, the prefix stored-energy envelope is:

$$
E_0-r\min(t,\gamma)
\le E(t)\le
E_0+r\min(t,\gamma).
$$

This displayed envelope explains the budget numerically; the independent deployability audit still evaluates the complete migrated restriction, including recovery-window, counterpart, integrality, terminal, and slack families.

In [ ]:
planning_hours = [(index + 1) * 0.25 for index in range(96)]
baseline_by_planning = [
    restriction_result.decisions.energy_baseline_charge_positive_kw[
        time_contract.planning_to_energy[index]
    ]
    for index in range(96)
]
reserve_by_planning = [
    restriction_result.decisions.fcr_capacity_kw[
        time_contract.planning_to_fcr[index]
    ]
    for index in range(96)
]
headroom_table = pd.DataFrame(
    {
        "planning hour end": planning_hours,
        "discharge edge p-r (kW)": [
            p - r for p, r in zip(baseline_by_planning, reserve_by_planning, strict=True)
        ],
        "discharge limit (kW)": [-battery.maximum_discharge_power_kw] * 96,
        "charge edge p+r (kW)": [
            p + r for p, r in zip(baseline_by_planning, reserve_by_planning, strict=True)
        ],
        "charge limit (kW)": [battery.maximum_charge_power_kw] * 96,
    }
)
flat_reserve_kw = restriction_result.decisions.fcr_capacity_kw[0]
assert all(value == flat_reserve_kw for value in restriction_result.decisions.fcr_capacity_kw)
stored_energy_envelope_table = pd.DataFrame(
    {
        "planning hour end": planning_hours,
        "lower horizon-budget envelope (kWh)": [
            battery.initial_stored_energy_kwh
            - flat_reserve_kw * min(hour, horizon_budget.horizon_activation_budget_hours)
            for hour in planning_hours
        ],
        "upper horizon-budget envelope (kWh)": [
            battery.initial_stored_energy_kwh
            + flat_reserve_kw * min(hour, horizon_budget.horizon_activation_budget_hours)
            for hour in planning_hours
        ],
        "battery minimum (kWh)": [battery.minimum_stored_energy_kwh] * 96,
        "battery maximum (kWh)": [battery.maximum_stored_energy_kwh] * 96,
    }
)
display(headroom_table.head(12), stored_energy_envelope_table.head(16), audit_table)

In [ ]:
robust_figure, robust_axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)
robust_axes[0].fill_between(
    stored_energy_envelope_table["planning hour end"],
    stored_energy_envelope_table["lower horizon-budget envelope (kWh)"],
    stored_energy_envelope_table["upper horizon-budget envelope (kWh)"],
    alpha=0.25,
    label="horizon-budget envelope",
)
robust_axes[0].plot(
    planning_hours,
    stored_energy_envelope_table["battery minimum (kWh)"],
    linestyle="--",
    label="battery minimum",
)
robust_axes[0].plot(
    planning_hours,
    stored_energy_envelope_table["battery maximum (kWh)"],
    linestyle="--",
    label="battery maximum",
)
robust_axes[0].set_ylabel("Stored energy (kWh)")
robust_axes[0].set_xlabel("Hours since delivery start")
robust_axes[0].set_title("Hand-calculated flat-reserve horizon-budget envelope")
robust_axes[0].legend()
robust_axes[1].plot(
    planning_hours,
    headroom_table["discharge edge p-r (kW)"],
    label="discharge edge p-r",
)
robust_axes[1].plot(
    planning_hours,
    headroom_table["charge edge p+r (kW)"],
    label="charge edge p+r",
)
robust_axes[1].plot(
    planning_hours,
    headroom_table["discharge limit (kW)"],
    linestyle="--",
    label="discharge limit",
)
robust_axes[1].plot(
    planning_hours,
    headroom_table["charge limit (kW)"],
    linestyle="--",
    label="charge limit",
)
robust_axes[1].set_ylabel("Power (kW, charge-positive)")
robust_axes[1].set_xlabel("Hours since delivery start")
robust_axes[1].set_title("Pointwise symmetric-power headroom")
robust_axes[1].legend(ncol=2)
generated_figures["robust_feasibility"] = robust_figure
robust_figure

## Ex-ante optimization objective

The source-frame bidding objective is assembled before delivery:

$$
z=
\Delta t\sum_k \pi^{E}_{h(k)}x_k^0
+\sum_b \pi^{FCR}_b x_b^r
-c_{\mathrm{slack}}\sum s.
$$

It is measured in EUR. It is not Feature 012's realized settlement or realized profit, even if this simple authored example later produces the same numerical total.

In [ ]:
objective_rows = []
for result in (restriction_result, relaxation_result):
    objective = result.raw_objective
    assert objective is not None
    objective_rows.extend(
        [
            (result.formulation.kind, "day-ahead energy value", objective.day_ahead_energy_value_eur, "EUR"),
            (result.formulation.kind, "FCR capacity value", objective.fcr_capacity_value_eur, "EUR"),
            (result.formulation.kind, "feasibility slack penalty", -objective.feasibility_slack_penalty_eur, "EUR"),
            (result.formulation.kind, "total objective", objective.total_objective_eur, "EUR"),
            (result.formulation.kind, "reconciliation residual", objective.reconciliation_residual_eur, "EUR"),
        ]
    )
objective_table = pd.DataFrame(
    objective_rows,
    columns=["formulation", "component", "value", "unit"],
)
objective_table

In [ ]:
objective_figure, objective_axis = plt.subplots(figsize=(10, 4), constrained_layout=True)
objective_labels = [
    "restriction objective",
    "restriction bound",
    "relaxation objective",
    "relaxation bound",
]
objective_values = [
    restriction_result.solver.objective_value_eur,
    restriction_result.solver.best_upper_bound_eur,
    relaxation_result.solver.objective_value_eur,
    relaxation_result.solver.best_upper_bound_eur,
]
objective_axis.bar(objective_labels, objective_values)
objective_axis.set_ylabel("Ex-ante value (EUR)")
objective_axis.set_title("Optimization objective and same-solve upper-bound evidence")
objective_axis.tick_params(axis="x", rotation=15)
generated_figures["objective_evidence"] = objective_figure
objective_figure

## Post-acceptance handoff to Feature 012

The eligible restriction plan is now fixed. Only at this point do we author a realized activation trace and accepted-volume evidence. The plan object and both schedule fingerprints pass unchanged into Feature 012; no Feature 013 battery transition, tracking, settlement, or profit model is introduced.

The activation pattern alternates upward discharge, downward charge, and zero at 15-minute resolution. It was absent from every preceding bid-selection cell.

In [ ]:
def build_walkthrough_evaluation_request(plan):
    period = plan.energy_schedule.covered_period
    evaluation_time = MultiRateTimeContract(
        start_utc=period.start_utc,
        end_utc=period.end_utc,
        activation_resolution="PT15M",
        state_checkpoint_resolution="PT15M",
        energy_resolution="PT1H",
        fcr_resolution="PT4H",
        market_timezone="Europe/Paris",
        interval_convention="start-inclusive/end-exclusive",
    )
    activation_values = tuple(
        (1.0, -1.0, 0.0, 0.0)[index % 4]
        for index in range(len(evaluation_time.index.activation_intervals))
    )
    activation_request = ActivationImportRequest(
        source_name="Feature 013 walkthrough post-acceptance activation",
        source_locator=(
            "examples/robust_fcr_bidding/"
            "robust_fcr_bidding_walkthrough.ipynb"
        ),
        source_revision="sha256:" + stable_fingerprint(list(activation_values)),
        source_timezone="Europe/Paris",
        requested_start_utc=period.start_utc,
        requested_end_utc=period.end_utc,
        source_resolution="PT15M",
        source_value_kind="interval_average",
        source_sign_meaning="positive source activation means upward/discharge",
        normalization_rule={
            "conversion": "identity",
            "formula": "activation = source_value",
            "point_to_interval": "already_interval_average",
            "clip_min": -1.0,
            "clip_max": 1.0,
            "target_sign": "positive is upward/discharge",
        },
        redistribution_basis={
            "source": "project-authored notebook input",
            "terms": "Apache-2.0",
        },
    )
    activation_trace = import_regulation_activation_trace(
        activation_request,
        [
            {
                "start_utc": interval.start_utc,
                "end_utc": interval.end_utc,
                "value": value,
                "source_value": value,
                "source_unit": "dimensionless",
            }
            for interval, value in zip(
                evaluation_time.index.activation_intervals,
                activation_values,
                strict=True,
            )
        ],
        trace_id="feature-013-walkthrough-activation-v1",
        created_at_utc=datetime(2026, 8, 29, 12, tzinfo=UTC),
    )
    day_prices_eur_per_mwh = tuple(
        value * 1000.0
        for value in walkthrough_problem.prices.day_ahead_energy_prices_eur_per_kwh
    )
    day_observations = tuple(
        EvaluationPriceObservation(
            interval=item.interval,
            price_kind="day_ahead_energy",
            source_value=price,
            source_unit="EUR/MWh",
            canonical_value=price,
            canonical_unit="EUR/MWh",
            conversion_id="walkthrough-eur-per-kwh-to-eur-per-mwh-v1",
            source_ref=f"feature_013_walkthrough_day_ahead:{index}",
        )
        for index, (item, price) in enumerate(
            zip(plan.energy_schedule.intervals, day_prices_eur_per_mwh, strict=True)
        )
    )
    fcr_observations = tuple(
        EvaluationPriceObservation(
            interval=item.interval,
            price_kind="fcr_capacity",
            source_value=price,
            source_unit="EUR/kW/block",
            canonical_value=price,
            canonical_unit="EUR/kW/block",
            conversion_id="walkthrough-identity-v1",
            source_ref=item.price_observation_ref,
        )
        for item, price in zip(
            plan.fcr_schedule.intervals,
            walkthrough_problem.prices.fcr_capacity_prices_eur_per_kw_block,
            strict=True,
        )
    )
    accepted = {
        item.volume_observation_ref: {
            "capacity_kw": item.capacity_kw,
            "unit": "kW",
            "evidence_kind": "authored_post_acceptance",
        }
        for item in plan.fcr_schedule.intervals
    }
    market_context = EvaluationMarketContext(
        context_kind="reference_result",
        semantics_version="feature-013-walkthrough-market-v1",
        period=CanonicalInterval.from_utc(period.start_utc, period.end_utc),
        day_ahead_prices=day_observations,
        fcr_capacity_prices=fcr_observations,
        accepted_bid_evidence=accepted,
        source_fingerprints={
            "post_acceptance_market": stable_fingerprint(
                {
                    "plan": plan.plan_fingerprint,
                    "day_prices_eur_per_mwh": day_prices_eur_per_mwh,
                    "fcr_prices": walkthrough_problem.prices.fcr_capacity_prices_eur_per_kw_block,
                }
            )
        },
    )
    replay_battery = BatteryComponent(
        name="feature-013-walkthrough-battery",
        capacity_kwh=battery.maximum_stored_energy_kwh,
        max_charge_kw=battery.maximum_charge_power_kw,
        max_discharge_kw=battery.maximum_discharge_power_kw,
        charge_efficiency=battery.charge_efficiency,
        discharge_efficiency=battery.discharge_efficiency,
        initial_soc=battery.initial_stored_energy_kwh / battery.maximum_stored_energy_kwh,
        min_soc=battery.minimum_stored_energy_kwh / battery.maximum_stored_energy_kwh,
        max_soc=1.0,
    )
    return FCRBidEvaluationRequest(
        evaluation_id="feature-013-walkthrough-evaluation-v1",
        plan=plan,
        market_context=market_context,
        activation_trace=activation_trace,
        battery=replay_battery,
        time_contract=evaluation_time,
        settlement_profile="robust-storage-bidding-reference-settlement-v1",
        penalty_rule=None,
        terminal_soc_target=None,
        comparison_fixture=None,
        numeric_tolerance=1e-6,
    )

walkthrough_evaluation_request = build_walkthrough_evaluation_request(
    restriction_result.deployable_plan
)

In [ ]:
walkthrough_evaluation_report = evaluate_fixed_fcr_bid(
    walkthrough_evaluation_request
)
walkthrough_evaluation_payload = walkthrough_evaluation_report.to_dict()
assert walkthrough_evaluation_report.status == "settled"

In [ ]:
replay_samples = walkthrough_evaluation_payload["physical_trace"]["samples"]
replay_trace_table = pd.DataFrame(
    [
        {
            "sample": index,
            "hours since start": index * 0.25,
            "activation": item["activation"],
            "baseline power (kW)": item["baseline_power_kw"],
            "requested combined power (kW)": item["requested_combined_power_kw"],
            "delivered combined power (kW)": item["delivered_combined_power_kw"],
            "stored energy after (kWh)": item["stored_energy_after_kwh"],
            "SOC after (%)": 100.0 * item["soc_after"],
            "absolute tracking error (kW)": item["absolute_tracking_error_kw"],
            "saturation": ", ".join(item["saturation_causes"]) or "none",
        }
        for index, item in enumerate(replay_samples)
    ]
)
physical_metrics = walkthrough_evaluation_payload["physical_metrics"]
physical_summary_table = pd.DataFrame(
    [
        ("requested power minimum", physical_metrics["requested_power_kw"]["minimum"], "kW"),
        ("requested power maximum", physical_metrics["requested_power_kw"]["maximum"], "kW"),
        ("stored energy minimum", physical_metrics["stored_energy_kwh"]["minimum"], "kWh"),
        ("stored energy maximum", physical_metrics["stored_energy_kwh"]["maximum"], "kWh"),
        ("stored energy terminal", physical_metrics["stored_energy_kwh"]["terminal"], "kWh"),
        ("throughput", physical_metrics["throughput_kwh"], "kWh"),
        ("tracking-error sum", physical_metrics["absolute_tracking_error_kw_sum"], "kW-samples"),
        ("saturation samples", physical_metrics["saturation_sample_count"], "count"),
    ],
    columns=["quantity", "value", "unit"],
)
display(replay_trace_table.head(16), physical_summary_table)

In [ ]:
replay_figure, replay_axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)
replay_axes[0].step(
    replay_trace_table["hours since start"],
    replay_trace_table["requested combined power (kW)"],
    where="post",
    label="requested",
)
replay_axes[0].step(
    replay_trace_table["hours since start"],
    replay_trace_table["delivered combined power (kW)"],
    where="post",
    linestyle="--",
    label="delivered",
)
replay_axes[0].set_ylabel("Battery power (kW, charge-positive)")
replay_axes[0].set_xlabel("Hours since delivery start")
replay_axes[0].set_title("Feature 012 requested and delivered power")
replay_axes[0].legend()
replay_axes[1].plot(
    replay_trace_table["hours since start"],
    replay_trace_table["stored energy after (kWh)"],
    label="stored energy",
    color="tab:green",
)
replay_axes[1].set_ylabel("Stored energy (kWh)")
replay_axes[1].set_xlabel("Hours since delivery start")
replay_axes[1].set_title("Feature 012 realized battery state")
soc_axis = replay_axes[1].twinx()
soc_axis.plot(
    replay_trace_table["hours since start"],
    replay_trace_table["SOC after (%)"],
    linestyle=":",
    color="tab:purple",
    label="SOC",
)
soc_axis.set_ylabel("State of charge (%)")
lines_left, labels_left = replay_axes[1].get_legend_handles_labels()
lines_right, labels_right = soc_axis.get_legend_handles_labels()
replay_axes[1].legend(lines_left + lines_right, labels_left + labels_right)
generated_figures["physical_replay"] = replay_figure
replay_figure

## Ex-post settlement remains separate

Feature 012 now combines day-ahead energy, accepted FCR capacity, realized activation-energy, and penalty components. All displayed monetary values use EUR. Unavailable intraday, imbalance, degradation, and terminal-value components stay explicitly unavailable rather than being silently treated as measured data.

In [ ]:
settlement_payload = walkthrough_evaluation_payload["settlement"]
settlement_components = (
    "day_ahead_revenue",
    "fcr_capacity_revenue",
    "activation_energy_revenue",
    "penalty",
    "intraday",
    "imbalance",
    "degradation",
    "terminal_value",
)
settlement_table = pd.DataFrame(
    [
        {
            "component": name,
            "status": settlement_payload[name]["status"],
            "value (EUR)": settlement_payload[name]["value_eur"],
            "reason": settlement_payload[name]["reason"],
        }
        for name in settlement_components
    ]
)
economics_separation_table = pd.DataFrame(
    [
        ("Feature 013 optimization objective", restriction_result.solver.objective_value_eur, "EUR", "ex-ante"),
        ("Feature 013 best upper bound", restriction_result.solver.best_upper_bound_eur, "EUR", "ex-ante"),
        ("Feature 012 realized settlement/profit", settlement_payload["total_profit_eur"], "EUR", "ex-post"),
        ("Feature 012 settlement reconciliation error", settlement_payload["reconciliation_error_eur"], "EUR", "ex-post audit"),
    ],
    columns=["quantity", "value", "unit", "information time"],
)
display(settlement_table, economics_separation_table)

In [ ]:
available_settlement = settlement_table[
    settlement_table["status"] == "available"
].copy()
settlement_figure, settlement_axis = plt.subplots(figsize=(10, 4), constrained_layout=True)
settlement_axis.bar(
    available_settlement["component"],
    available_settlement["value (EUR)"],
)
settlement_axis.axhline(0.0, color="black", linewidth=0.8)
settlement_axis.set_ylabel("Realized settlement component (EUR)")
settlement_axis.set_title(
    f"Feature 012 realized total profit: {settlement_payload['total_profit_eur']:.3f} EUR"
)
settlement_axis.tick_params(axis="x", rotation=20)
generated_figures["realized_settlement"] = settlement_figure
settlement_figure

## Repeatability and identity

Runtime duration is volatile and is not used as semantic identity. The problem, decision vector, plan, solver-result, physical trace, settlement, and final report each retain distinct fingerprints so that an optimization result cannot be confused with a later replay.

In [ ]:
fingerprint_table = pd.DataFrame(
    [
        ("bidding problem", walkthrough_problem.problem_fingerprint),
        ("restriction semantic result", restriction_result.semantic_result_fingerprint),
        ("relaxation semantic result", relaxation_result.semantic_result_fingerprint),
        ("bound comparison", bound_comparison.comparison_fingerprint),
        ("restriction decision vector", restriction_result.decisions.decision_vector_fingerprint),
        ("deployable plan", restriction_result.deployable_plan.plan_fingerprint),
        ("energy schedule", restriction_result.deployable_plan.energy_schedule.schedule_fingerprint),
        ("FCR schedule", restriction_result.deployable_plan.fcr_schedule.schedule_fingerprint),
        ("activation trace", walkthrough_evaluation_request.activation_trace.artifact_fingerprint),
        ("physical trace", walkthrough_evaluation_report.trace_fingerprint),
        ("settlement", walkthrough_evaluation_report.settlement_fingerprint),
        ("evaluation result", walkthrough_evaluation_report.result_fingerprint),
    ],
    columns=["artifact", "semantic fingerprint"],
)
fingerprint_table

## Offline Julia/Gurobi fixture: what it can and cannot prove

The normalized 2023-06-18 Experiment 2 fixture records the pinned reference revision, input profile, displayed objective and bound, 24 hourly source-frame decisions, and six FCR decisions. The source JSON rounds numeric results to three decimals.

For a displayed value $v$, the governed closed source-precision envelope is:

$$
[v-0.0005,\;v+0.0005].
$$

This is a rounding interval in the field's physical unit (kW or EUR), not an allowed relative error such as “0.36 times.” Feasibility still requires the independent restriction audit; the saved source field bilinear_violations=0 is not feasibility proof.

In [ ]:
reference_fixture = load_robust_fcr_reference_fixture(
    REFERENCE_FIXTURE_PATH
)
reference_decision_intervals = build_reference_decision_intervals(
    reference_fixture
)

In [ ]:
reference_provenance_table = pd.DataFrame(
    [
        ("fixture id", reference_fixture.fixture_id),
        ("fixture fingerprint", reference_fixture.fixture_fingerprint),
        ("reference revision", reference_fixture.reference_provenance["reference_revision"]),
        ("raw result SHA-256", reference_fixture.reference_provenance["original_result_sha256"]),
        ("raw result bytes", reference_fixture.reference_provenance["original_result_byte_count"]),
        ("recorded objective (EUR)", reference_fixture.recorded_result.objective_value_eur),
        ("recorded bound (EUR)", reference_fixture.recorded_result.objective_bound_eur),
        ("saved decimal places", reference_fixture.recorded_result.saved_decimal_places),
    ],
    columns=["field", "value"],
)
reference_precision_table = pd.DataFrame(
    [
        ("decision half-unit", reference_fixture.precision_policy.decision_half_unit_kw, "kW"),
        ("objective half-unit", reference_fixture.precision_policy.objective_half_unit_eur, "EUR"),
        ("deployability threshold", reference_fixture.precision_policy.deployability_threshold, "absolute canonical unit"),
        ("interval closure", reference_fixture.precision_policy.interval_closure, "category"),
        ("certification method", reference_fixture.precision_policy.certification_method, "category"),
    ],
    columns=["quantity", "value", "unit"],
)
reference_unavailable_table = pd.DataFrame(
    [item.to_dict() for item in reference_fixture.unavailable_fields]
)
reference_decision_table = pd.DataFrame(
    [item.to_dict() for item in reference_decision_intervals]
)
display(
    reference_provenance_table,
    reference_precision_table,
    reference_unavailable_table,
    reference_decision_table,
)

In [ ]:
source_energy = reference_decision_table[
    reference_decision_table["decision_kind"] == "energy_x0"
]
source_fcr = reference_decision_table[
    reference_decision_table["decision_kind"] == "fcr_xr"
]
reference_figure, reference_axes = plt.subplots(2, 1, figsize=(10, 7), constrained_layout=True)
reference_axes[0].plot(
    source_energy["index"],
    source_energy["displayed_value"],
    marker="o",
    label="displayed Julia x0",
)
reference_axes[0].fill_between(
    source_energy["index"],
    source_energy["lower"],
    source_energy["upper"],
    alpha=0.35,
    label="closed source-precision envelope",
)
reference_axes[0].set_ylabel("Energy decision (kW, source discharge +)")
reference_axes[0].set_xlabel("Delivery hour")
reference_axes[0].set_title("Recorded Julia hourly decisions and physical-unit precision")
reference_axes[0].legend()
reference_axes[1].plot(
    source_fcr["index"],
    source_fcr["displayed_value"],
    marker="o",
    label="displayed Julia xr",
)
reference_axes[1].fill_between(
    source_fcr["index"],
    source_fcr["lower"],
    source_fcr["upper"],
    alpha=0.35,
    label="closed source-precision envelope",
)
reference_axes[1].set_ylabel("Symmetric FCR decision (kW)")
reference_axes[1].set_xlabel("Four-hour block")
reference_axes[1].set_title("Recorded Julia FCR decisions and physical-unit precision")
reference_axes[1].legend()
generated_figures["reference_source_precision"] = reference_figure
reference_figure

## Optional full native 2023-06-18 benchmark

Set RUN_NATIVE_REFERENCE_BENCHMARK to True only for an explicit research run. The complete 96-step reference restriction can consume the full configured time limit. It uses the committed normalized fixture, never reads the baseline tree, and reports comparison categories field by field.

A match at source precision is strong bounded reproduction evidence. It is still not a claim of exact Julia parity unless unrounded decisions, common feasibility, and sufficient optimality evidence all exist.

In [ ]:
native_reference_benchmark = {
    "status": "not_requested",
    "reason": "RUN_NATIVE_REFERENCE_BENCHMARK is False",
}
native_reference_comparison_table = pd.DataFrame()

if RUN_NATIVE_REFERENCE_BENCHMARK:
    benchmark_policy = RobustFCRSolvePolicy(
        solver_name="highs",
        time_limit_seconds=REFERENCE_BENCHMARK_TIME_LIMIT_SECONDS,
        threads=1,
        random_seed=13,
        primal_feasibility_tolerance=1e-7,
        dual_feasibility_tolerance=1e-7,
        mip_feasibility_tolerance=1e-6,
        relative_mip_gap=1e-6,
        objective_comparison_absolute_tolerance_eur=1e-6,
        objective_comparison_relative_tolerance=1e-9,
        allow_feasible_incumbent_plan=False,
    )
    benchmark_problem = build_robust_fcr_reference_problem(
        reference_fixture,
        solve_policy=benchmark_policy,
    )
    benchmark_result = solve_robust_fcr_bid(
        benchmark_problem,
        formulation="restriction",
    )
    native_reference_benchmark = {
        "status": benchmark_result.solver.solve_status,
        "termination": benchmark_result.solver.termination_status,
        "objective_eur": benchmark_result.solver.objective_value_eur,
        "best_upper_bound_eur": benchmark_result.solver.best_upper_bound_eur,
        "deployability": benchmark_result.deployability_status,
        "result_fingerprint": benchmark_result.semantic_result_fingerprint,
    }
    if benchmark_result.decisions is not None:
        native_reference_comparison = compare_robust_fcr_reference(
            reference_fixture,
            benchmark_result,
        )
        native_reference_comparison_table = pd.DataFrame(
            [field.to_dict() for field in native_reference_comparison.field_results]
        )
        native_reference_benchmark["comparison_status"] = (
            native_reference_comparison.overall_status
        )
        native_reference_benchmark["comparison_fingerprint"] = (
            native_reference_comparison.comparison_fingerprint
        )

display(native_reference_benchmark, native_reference_comparison_table)

## Optional licensed Gurobi exact evidence

Set RUN_OPTIONAL_GUROBI_EXACT to True only after installing the optional Gurobi extra and configuring a valid license. The exact bilinear formulation is validation evidence, never a deployable market plan. When available, the checked ordering is:

$$
z_{\mathrm{restriction}}
\le z_{\mathrm{exact}}
\le z_{\mathrm{relaxation}}.
$$

Missing exact evidence remains explicitly unavailable and does not weaken the routine HiGHS check.

In [ ]:
optional_exact_result = {
    "status": "not_requested",
    "reason": "RUN_OPTIONAL_GUROBI_EXACT is False",
}
optional_exact_table = pd.DataFrame()

if RUN_OPTIONAL_GUROBI_EXACT:
    gurobi_policy = RobustFCRSolvePolicy(
        solver_name="gurobi",
        time_limit_seconds=OPTIONAL_GUROBI_TIME_LIMIT_SECONDS,
        threads=1,
        random_seed=13,
        primal_feasibility_tolerance=1e-7,
        dual_feasibility_tolerance=1e-7,
        mip_feasibility_tolerance=1e-6,
        relative_mip_gap=1e-6,
        objective_comparison_absolute_tolerance_eur=1e-6,
        objective_comparison_relative_tolerance=1e-9,
        allow_feasible_incumbent_plan=False,
    )
    gurobi_problem = replace(
        walkthrough_problem,
        solve_policy=gurobi_policy,
    )
    gurobi_restriction = solve_robust_fcr_bid(
        gurobi_problem,
        formulation="restriction",
    )
    gurobi_relaxation = solve_robust_fcr_bid(
        gurobi_problem,
        formulation="relaxation",
    )
    gurobi_exact = solve_robust_fcr_bid(
        gurobi_problem,
        formulation="exact",
    )
    optional_exact_comparison = compare_robust_fcr_bounds(
        gurobi_restriction,
        gurobi_relaxation,
        exact=gurobi_exact,
    )
    optional_exact_result = {
        "status": optional_exact_comparison.overall_status,
        "exact_status": optional_exact_comparison.exact_status,
        "comparison_fingerprint": optional_exact_comparison.comparison_fingerprint,
    }
    optional_exact_table = pd.DataFrame(
        [
            {
                "formulation": result.formulation.kind,
                "solve status": result.solver.solve_status,
                "objective (EUR)": result.solver.objective_value_eur,
                "best upper bound (EUR)": result.solver.best_upper_bound_eur,
                "deployability": result.deployability_status,
                "plan emitted": result.deployable_plan is not None,
            }
            for result in (gurobi_restriction, gurobi_exact, gurobi_relaxation)
        ]
    )

display(optional_exact_result, optional_exact_table)

## Optional explicit-path reference activation replay

Set RUN_EXTERNAL_ACTIVATION_REPLAY to True only when the governed external CSV is installed at the explicit configured path. This section replays the recorded fixed reference plan through Feature 012; it does not expose activation to the Feature 013 optimizer and does not search for, download, infer, or substitute the file.

The complete native-versus-recorded external replay remains the selected external_reference_data workflow documented in the Feature 013 quickstart because it may require a separate 600-second native solve.

In [ ]:
external_activation_result = {
    "status": "not_requested",
    "reason": "RUN_EXTERNAL_ACTIVATION_REPLAY is False",
}

if RUN_EXTERNAL_ACTIVATION_REPLAY:
    if not EXTERNAL_ACTIVATION_PATH.is_file():
        raise FileNotFoundError(
            "explicit reference activation CSV is absent at "
            f"{EXTERNAL_ACTIVATION_PATH}; install the governed file or disable "
            "RUN_EXTERNAL_ACTIVATION_REPLAY"
        )
    from examples.fcr_bid_evaluation.run_reference_parity import (
        build_reference_request,
    )
    external_request = build_reference_request(
        fixture_path=EVALUATION_FIXTURE_PATH,
        activation_path=EXTERNAL_ACTIVATION_PATH,
    )
    external_report = evaluate_fixed_fcr_bid(external_request)
    external_activation_result = {
        "status": external_report.status,
        "activation_count": len(external_request.activation_trace.samples),
        "trace_fingerprint": external_report.trace_fingerprint,
        "settlement_fingerprint": external_report.settlement_fingerprint,
        "result_fingerprint": external_report.result_fingerprint,
        "total_profit_eur": external_report.total_profit_eur,
    }

external_activation_result

In [ ]:
exported_figure_paths = []
if EXPORT_FIGURES:
    FIGURE_EXPORT_DIR.mkdir(parents=True, exist_ok=True)
    for figure_name, figure in generated_figures.items():
        destination = FIGURE_EXPORT_DIR / f"{figure_name}.png"
        figure.savefig(destination, dpi=160, bbox_inches="tight")
        exported_figure_paths.append(destination)

exported_figure_paths

In [ ]:
notebook_display_payload = {
    "problem_fingerprint": walkthrough_problem.problem_fingerprint,
    "restriction_result_fingerprint": restriction_result.semantic_result_fingerprint,
    "relaxation_result_fingerprint": relaxation_result.semantic_result_fingerprint,
    "bound_comparison_fingerprint": bound_comparison.comparison_fingerprint,
    "decision_vector_fingerprint": restriction_result.decisions.decision_vector_fingerprint,
    "plan_fingerprint": restriction_result.deployable_plan.plan_fingerprint,
    "physical_trace_fingerprint": walkthrough_evaluation_report.trace_fingerprint,
    "settlement_fingerprint": walkthrough_evaluation_report.settlement_fingerprint,
    "evaluation_result_fingerprint": walkthrough_evaluation_report.result_fingerprint,
    "reference_fixture_fingerprint": reference_fixture.fixture_fingerprint,
    "reference_decision_content_fingerprint": (
        reference_fixture.recorded_result.decision_content_fingerprint
    ),
    "generated_figure_names": sorted(generated_figures),
}
notebook_display_fingerprint = stable_fingerprint(notebook_display_payload)
pd.DataFrame(
    [
        ("notebook display fingerprint", notebook_display_fingerprint),
        ("default figure count", len(generated_figures)),
        ("native reference benchmark", native_reference_benchmark["status"]),
        ("optional exact evidence", optional_exact_result["status"]),
        ("external activation replay", external_activation_result["status"]),
    ],
    columns=["evidence", "value"],
)

## Conclusions

- Feature 013 can solve one ordinary 24/96/6 joint energy/FCR problem with native Pyomo/HiGHS while keeping all realized activation outside bid selection.
- The restriction alone emitted an audited robust-labelled FixedFCRBidPlan. The relaxation remained upper-bound evidence and emitted no plan.
- Feature 012 accepted the unchanged restriction plan and remained the sole owner of physical replay, tracking, stored energy, settlement, and realized profit.
- Stable semantic fingerprints distinguish problem, decisions, plan, trace, settlement, and final evaluation evidence.
- The normalized Julia/Gurobi fixture supports provenance-rich, source-precision-bounded comparison. Exact unrounded schedule equality, source optimality proof, and complete Julia trace parity are not available from that artifact.

Rolling or multi-day studies, SOC carry-over, intraday recourse, market-acceptance proof, TPM fitting or bid selection, degradation costs, aFRR/mFRR, live data acquisition, and full-history paper experiments remain out of scope. Those boundaries are evidence limits, not omitted values to infer.